# 00 | Trích đặc trưng V6 và dựng tập chia theo bệnh nhân

Từ tín hiệu thô sinh ra `data/features/v6/` (14 đặc trưng: 13 HRV + PPG_AC) và `data/splits/` (DEV/TEST theo bệnh nhân của PulseWatch, Liu; manifest DeepBeat). Mỗi ô gọi một script trong `notebooks/v6/scripts/`, chạy đúng thứ tự.

| Cần có trước | Từ đâu |
|---|---|
| `data/raw/deepbeat/`, `data/raw/pulsewatch/`, `data/raw/liu2022/`, `data/raw/mimic_perform/` | tải theo README.md mục 8 |
| `results/v5/pulsewatch_frozen_v5_features.csv`, `liu_pseudo30_frozen_v5_predictions.csv`, `peakdet_pac_pvc_frozen_v5_predictions.csv` | `notebooks/v5/02_external_validation` |

`data/features/v6/`, `v7/` được commit; `data/splits/` và `data/cache/` thì không (thoả thuận sử dụng dữ liệu Synapse). SHA-256 ghi ở `data/MANIFEST.csv`.

In [ ]:
import os, sys, time, subprocess, glob
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_ml").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Không tìm thấy gốc repo")
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

def run(script, *args):
    """Chạy một script từ gốc repo bằng đúng Python của kernel; in log; dừng nếu lỗi."""
    t0 = time.time()
    print(f"\n$ python {script} {' '.join(args)}", flush=True)
    r = subprocess.run([sys.executable, script, *args], cwd=ROOT, text=True, encoding="utf-8", errors="replace",
                       capture_output=True, env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONPATH": "src"})
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:]); raise RuntimeError(f"{script} lỗi, mã {r.returncode}")
    print(f"  xong trong {time.time()-t0:.0f}s")

## 1. Cache tín hiệu DeepBeat dạng memmap

In [ ]:
run("notebooks/v6/scripts/prepare_deepbeat_signal_memmap.py")

## 2. Manifest cửa sổ DeepBeat (train / validate / test độc lập)

In [ ]:
run("notebooks/v6/scripts/build_deepbeat_v6_manifest.py")

## 3. Trích 14 đặc trưng cho DeepBeat và MIMIC

In [ ]:
run("notebooks/v6/scripts/extract_deepbeat_v6_features.py")

## 4. Chia DEV/TEST theo bệnh nhân cho PulseWatch và Liu

In [ ]:
run("notebooks/v6/scripts/build_hard_domain_subject_splits.py")

## 5. Bảng hard negative PAC/PVC (v2)

In [ ]:
run("notebooks/v6/scripts/build_v7_hard_negative_v2.py")

## Kiểm tra đầu ra

In [ ]:
for f in sorted(glob.glob("data/features/v6/*.csv")) + sorted(glob.glob("data/splits/*")) + sorted(glob.glob("data/features/v7/hard_negative_v7_v2.csv")):
    print(f"{os.path.getsize(f)/1e6:8.2f} MB  {f}")